In [71]:
import pandas as pd
from datetime import datetime, timedelta
import numpy as np

import warnings
warnings.filterwarnings('ignore')
# Load the training dataset
file_path = "Phase 1 Prediction Output_original.xlsx"

# Load all sheets from the Excel file
xls = pd.ExcelFile(file_path)
xls_t = pd.ExcelFile("Phase 1 Training Dataset.xlsx")
sheet_names = xls.sheet_names
# Read all sheets into a dictionary of dataframes
nh_data = {sheet: xls.parse(sheet) for sheet in sheet_names}

nh_data_2 = {sheet: xls_t.parse(sheet) for sheet in sheet_names}

In [80]:
display(nh_data['Group 1'])


,Date,CNA Point Prediction,CNA Lower Bound,CNA Upper Bound,LPN Point Prediction,LPN Lower Bound,LPN Upper Bound,RN Point Prediction,RN Lower Bound,RN Upper Bound
0,2024-04-01,143.548004,90.985001,200.289999,76.103996,32.793000,112.860000,27.314001,18.267000,38.865000
1,2024-04-02,153.813995,86.270003,241.471001,78.702003,37.400999,110.485000,32.491997,18.691999,57.383000
2,2024-04-03,145.824005,82.435001,227.003000,73.007996,34.016000,108.565000,33.493999,22.768000,48.429000
3,2024-04-04,153.352005,95.232000,230.074001,67.581993,27.084999,91.784000,35.667999,13.897000,49.650000
4,2024-04-05,144.095993,104.267000,206.481000,72.225998,23.873999,98.140000,38.627998,24.691000,61.950000
...,...,...,...,...,...,...,...,...,...,...
86,2024-06-26,152.033295,94.999202,255.867145,71.539284,57.779803,102.062445,27.959814,12.625947,37.175045
87,2024-06-27,152.019608,97.516978,252.741130,69.226974,55.261053,97.370760,26.703613,11.612861,34.907005
88,2024-06-28,152.746185,97.359353,259.664183,67.942192,47.583870,99.113870,25.702778,10.017393,35.567190
89,2024-06-29,147.792511,96.463602,239.738855,64.312057,47.456295,90.857602,24.063223,11.484583,37.580300


In [66]:
"""Load training data from Excel file with multiple sheets"""
file_path = "./Phase 1 Training Dataset.xlsx"
xl = pd.ExcelFile(file_path)
groups = xl.sheet_names
train_df = pd.DataFrame()
test_df = pd.DataFrame()
for sheet in groups:
    df = pd.read_excel(file_path, sheet_name=sheet)
    
    # Skip the first row which contains staff type labels
    df = df.iloc[1:]
    
    # Convert the date column to datetime
    df['Date'] = pd.to_datetime(df.iloc[:, 0])
    
    # Sort by date to ensure temporal order
    df = df.sort_values('Date')
    
    # Calculate split point (80% train, 20% test)
    split_idx = int(len(df) * 0.8)
    
    for i in range(5):  # 5 NHs per group
        start_col = i * 4  # Each NH has 4 columns (NH No., CNA, LPN, RN)
        if start_col + 3 >= len(df.columns):
            break
            
        # Extract data for each staff type
        cna_data = pd.to_numeric(df.iloc[:, start_col + 1], errors='coerce')
        lpn_data = pd.to_numeric(df.iloc[:, start_col + 2], errors='coerce')
        rn_data = pd.to_numeric(df.iloc[:, start_col + 3], errors='coerce')        

        num_rows = len(df['Date'])
        # Add the columns in long format to the train and test dataframes
        aux_df_cna =  pd.DataFrame({
            'unique_id': [f"{sheet} NH No. {i + 1} CNA" for _ in range(num_rows)],
            'ds': df['Date'].dropna().tolist(),
            'y': cna_data.dropna().tolist()
        })
        aux_df_lpn =  pd.DataFrame({
            'unique_id': [f"{sheet} NH No. {i + 1} LPN" for _ in range(num_rows)],
            'ds': df['Date'].dropna().tolist(),
            'y': lpn_data.dropna().tolist()
        })
        aux_df_rn = pd.DataFrame({
            'unique_id': [f"{sheet} NH No. {i + 1} RN" for _ in range(num_rows)],
            'ds': df['Date'].dropna().tolist(),
            'y': rn_data.dropna().tolist()
        })
        train_df = pd.concat([train_df, aux_df_cna[:split_idx], aux_df_lpn[:split_idx], aux_df_rn[:split_idx]], ignore_index=True)
        test_df = pd.concat([test_df, aux_df_cna[split_idx:], aux_df_lpn[split_idx:], aux_df_rn[split_idx:]], ignore_index=True)


In [85]:
Y_df = pd.concat([train_df, test_df], ignore_index=True)

y_true = Y_df['y'].to_numpy()
y_pred = np.array([])
lower_bound = np.array([])
upper_bound = np.array([])
for k,v in nh_data.items():
    y_pred = np.concatenate((y_pred, np.repeat(v['CNA Point Prediction'].values, 5)))
    y_pred = np.concatenate((y_pred, np.repeat(v['LPN Point Prediction'].values, 5)))
    y_pred = np.concatenate((y_pred, np.repeat(v['RN Point Prediction'].values, 5)))
    lower_bound = np.concatenate((lower_bound, np.repeat(v['CNA Lower Bound'].values, 5)))
    lower_bound = np.concatenate((lower_bound, np.repeat(v['LPN Lower Bound'].values, 5)))
    lower_bound = np.concatenate((lower_bound, np.repeat(v['RN Lower Bound'].values, 5)))
    upper_bound = np.concatenate((upper_bound, np.repeat(v['CNA Upper Bound'].values, 5)))
    upper_bound = np.concatenate((upper_bound, np.repeat(v['LPN Upper Bound'].values, 5)))
    upper_bound = np.concatenate((upper_bound, np.repeat(v['RN Upper Bound'].values, 5)))
print(y_true)
print(y_pred)
print(lower_bound)
print(upper_bound)

[87.72 82.48 78.47 ... 12.75  8.    8.  ]
[143.54800415 143.54800415 143.54800415 ...  35.44799042  35.44799042
  35.44799042]
[90.98500137 90.98500137 90.98500137 ...  8.01742573  8.01742573
  8.01742573]
[200.28999939 200.28999939 200.28999939 ...  49.63154907  49.63154907
  49.63154907]


In [86]:
print(len(y_true))
print(len(y_pred))
print(len(lower_bound))
print(len(upper_bound))

27300
27300
27300
27300


In [90]:
def calculate_mae(y_true, y_pred):
    """Calculate Mean Absolute Error"""
    return np.mean(np.abs(y_true - y_pred))

def calculate_mape(y_true, y_pred):
    """Calculate Mean Absolute Percentage Error with handling for small values"""
    # Use a threshold to avoid division by very small numbers
    threshold = 1.0
    mask = y_true > threshold
    if not np.any(mask):
        return np.nan
    return 100 * np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]))

def calculate_smape(y_true, y_pred):
    """Calculate Symmetric Mean Absolute Percentage Error"""
    return 100 * np.mean(2.0 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred)))

def calculate_mis(y_true, y_pred, lower_bound, upper_bound):
    """Calculate Mean Interval Score with robust handling of outliers"""
    alpha = 0.05
    n = len(y_true)
    
    # Initialize components of MIS
    coverage_penalty = np.zeros(n)
    width_penalty = np.zeros(n)
    
    # Calculate penalties with outlier-robust handling
    for i in range(n):
        interval_width = upper_bound[i] - lower_bound[i]
        
        if y_true[i] < lower_bound[i]:
            coverage_penalty[i] = 2/alpha * (lower_bound[i] - y_true[i])
        elif y_true[i] > upper_bound[i]:
            coverage_penalty[i] = 2/alpha * (y_true[i] - upper_bound[i])
            
        width_penalty[i] = interval_width
        
    # Use median instead of mean for more robustness
    mis = np.median(width_penalty + coverage_penalty)
    return mis

def evaluate_predictions(y_true, y_pred, lower_bound, upper_bound):
    """Evaluate predictions using all metrics"""
    metrics = {
        'MAE': calculate_mae(y_true, y_pred),
        'MAPE': calculate_mape(y_true, y_pred),
        'SMAPE': calculate_smape(y_true, y_pred),
        'MIS': calculate_mis(y_true, y_pred, lower_bound, upper_bound)
    }
    return metrics
    
evaluate_predictions(y_true, y_pred, lower_bound, upper_bound)

{'MAE': 78.11353414869554,
 'MAPE': 205.96788239979244,
 'SMAPE': 87.09272728630454,
 'MIS': 1015.3516714096068}